# 1. Setup Paths and Data Sources 

Define imports, workspace paths, and the source locations.

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DAIOE_SOURCE: str = (
    "https://raw.githubusercontent.com/joseph-data/07_translate_ssyk/main/"
    "03_translated_files/daioe_ssyk2012_translated.csv"
)
SCB_SOURCE: str = (
    "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_years_regions/daioe_pull/"
    "data/processed/ssyk12_aggregated_ssyk4_to_ssyk1.parquet"
)



# 2. Load DAIOE and SCB Lazily

Read source files as `LazyFrame` objects for efficient transformations

In [2]:
daioe_lazy_lf = pl.scan_csv(
    DAIOE_SOURCE,
)

scb_lazy_lf = pl.scan_parquet(
    SCB_SOURCE,
)

# 3. Quick Sanity Checks

Preview data 

In [3]:
print(daioe_lazy_lf.head(5).collect())

shape: (5, 27)
┌────────────┬──────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ ssyk2012_4 ┆ year ┆ daioe_alla ┆ daioe_stra ┆ … ┆ pctl_rank_ ┆ ssyk2012_ ┆ ssyk2012_ ┆ ssyk2012_ │
│ ---        ┆ ---  ┆ pps        ┆ tgames     ┆   ┆ genai      ┆ 1         ┆ 2         ┆ 3         │
│ str        ┆ i64  ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆      ┆ f64        ┆ f64        ┆   ┆ f64        ┆ str       ┆ str       ┆ str       │
╞════════════╪══════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 0110 Commi ┆ 2010 ┆ null       ┆ null       ┆ … ┆ null       ┆ 0 Armed   ┆ 01        ┆ 011 Commi │
│ ssioned    ┆      ┆            ┆            ┆   ┆            ┆ forces    ┆ Officers  ┆ ssioned   │
│ armed      ┆      ┆            ┆            ┆   ┆            ┆ occupatio ┆           ┆ armed     │
│ forces…    ┆      ┆            ┆            ┆   ┆            ┆ ns        ┆

In [4]:
print(scb_lazy_lf.head(5).collect())

shape: (5, 8)
┌───────┬───────────┬────────────────────┬─────────────┬────────────────────┬───────┬──────┬───────┐
│ level ┆ ssyk_code ┆ occupation         ┆ county_code ┆ county             ┆ sex   ┆ year ┆ value │
│ ---   ┆ ---       ┆ ---                ┆ ---         ┆ ---                ┆ ---   ┆ ---  ┆ ---   │
│ str   ┆ str       ┆ str                ┆ str         ┆ str                ┆ str   ┆ i64  ┆ i64   │
╞═══════╪═══════════╪════════════════════╪═════════════╪════════════════════╪═══════╪══════╪═══════╡
│ SSYK4 ┆ 2343      ┆ Preschool teachers ┆ 24          ┆ Västerbotten       ┆ men   ┆ 2024 ┆ 114   │
│       ┆           ┆                    ┆             ┆ county             ┆       ┆      ┆       │
│ SSYK4 ┆ 4113      ┆ Back office-staff  ┆ 17          ┆ Värmland county    ┆ women ┆ 2017 ┆ 6     │
│ SSYK4 ┆ 5311      ┆ Child care workers ┆ 18          ┆ Örebro county      ┆ women ┆ 2021 ┆ 2377  │
│ SSYK4 ┆ 4212      ┆ Debt-collectors,   ┆ 05          ┆ Östergötland       ┆

# 4. Compute the 1-Yr, 3-Yr and 5-Yr Change

Aggregate to employment totals by `level`, `ssyk_code`, `occupation`, `county_code`, `sex`, and `year`, then compute absolute and percent changes over 1, 3, and 5 years using window shifts.

In [5]:
change_keys = [col for col in scb_lazy_lf.collect_schema().names() if col != "value"]
time_col = "year"
group_keys = [col for col in change_keys if col != time_col]

scb_lazy_lf_changes = (
    scb_lazy_lf
    .group_by(change_keys)
    .agg(pl.col("value").sum().alias("emp_count"))
    # Pre-compute shifted values once each
    .with_columns([
        pl.col("emp_count").shift(1).over(group_keys, order_by=time_col).alias("_emp_1y"),
        pl.col("emp_count").shift(3).over(group_keys, order_by=time_col).alias("_emp_3y"),
        pl.col("emp_count").shift(5).over(group_keys, order_by=time_col).alias("_emp_5y"),
    ])
    # Reuse pre-computed shifts
    .with_columns([
        (pl.col("emp_count") - pl.col("_emp_1y")).alias("chg_1y"),
        (pl.col("emp_count") - pl.col("_emp_3y")).alias("chg_3y"),
        (pl.col("emp_count") - pl.col("_emp_5y")).alias("chg_5y"),
        ((pl.col("emp_count") / pl.col("_emp_1y") - 1) * 100).alias("pct_chg_1y"),
        ((pl.col("emp_count") / pl.col("_emp_3y") - 1) * 100).alias("pct_chg_3y"),
        ((pl.col("emp_count") / pl.col("_emp_5y") - 1) * 100).alias("pct_chg_5y"),
    ])
    .drop("_emp_1y", "_emp_3y", "_emp_5y")
)

scb_lazy_lf_changes.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('occupation', String),
        ('county_code', String),
        ('county', String),
        ('sex', String),
        ('year', Int64),
        ('emp_count', Int64),
        ('chg_1y', Int64),
        ('chg_3y', Int64),
        ('chg_5y', Int64),
        ('pct_chg_1y', Float64),
        ('pct_chg_3y', Float64),
        ('pct_chg_5y', Float64)])

# 5. Derive SSYK2012 Levels in the DAIOE dataset

Split DAIOE SSYK4 into SSYK1-4 and keep SSYK2012 years.

In [6]:
SSYK12_START_YEAR = 2014  # First year of SSYK2012 publication

daioe_lazy_lf_ssyk12 = (
    daioe_lazy_lf
    .with_columns(
        pl.col("ssyk2012_4").str.slice(0, i).alias(f"code_{i}")
        for i in range(1, 5)
    )
    .drop(pl.col("^ssyk2012.*$"))
    .filter(pl.col("year") >= SSYK12_START_YEAR)
)
daioe_lazy_lf_ssyk12.limit(10).collect()


year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
2014,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2015,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2016,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2017,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2018,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2019,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2020,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2021,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2022,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""


# 6. Align DAIOE Years to SCB Coverage

Extend DAIOE series forward when SCB has later years.

In [7]:
## Here I extend the years to Latest according to the pulled SCB data (2024, yearly)

base = daioe_lazy_lf_ssyk12

# Extend years to latest available in SCB data (2024, yearly)
DAIOE_MAX_YEAR = base.select(pl.max("year")).collect().item()
SCB_MAX_YEAR = scb_lazy_lf.select(pl.max("year")).collect().item()
missing_years = list(range(DAIOE_MAX_YEAR + 1, SCB_MAX_YEAR + 1))

daioe_lazy_lf_extended = (
    base
    if not missing_years
    else pl.concat(
        [
            base,
            base
            .filter(pl.col("year") == DAIOE_MAX_YEAR)
            .drop("year")
            .join(pl.LazyFrame({"year": missing_years}), how="cross")
            .select(base.collect_schema().names()),
        ],
        how="vertical",
    )
)

In [8]:
daioe_lazy_lf_extended.filter(pl.col("year") == SCB_MAX_YEAR).limit(10).collect()

year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""02""","""021""","""0210"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""03""","""031""","""0310"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1""","""11""","""111""","""1111"""
2024,31.355038,0.25465,3.619455,0.402864,0.224535,0.739058,0.423185,1.027839,0.040984,0.735677,2.899789,47.869999,42.200001,7.21,42.91,54.02,62.77,59.93,57.330002,56.619999,55.439999,62.529999,"""1""","""11""","""111""","""1112"""
2024,33.85648,0.269727,3.7854,0.439125,0.247134,0.788555,0.461196,1.130982,0.04624,0.819611,3.146769,59.93,58.040001,11.7,58.509998,65.370003,72.93,68.910004,68.440002,70.330002,66.550003,70.57,"""1""","""11""","""111""","""1113"""
2024,32.331161,0.260123,3.675521,0.405983,0.23089,0.748559,0.441446,1.077137,0.043876,0.78386,2.993233,51.889999,50.0,8.16,44.330002,56.380001,64.660004,65.599998,63.950001,62.77,58.98,64.889999,"""1""","""11""","""112""","""1120"""
2024,38.505562,0.333748,4.106049,0.511271,0.285375,0.847376,0.551517,1.357987,0.05216,0.922529,3.600155,82.150002,89.239998,21.870001,81.68,84.989998,82.389999,87.830002,88.529999,84.75,82.620003,86.169998,"""1""","""12""","""121""","""1211"""
2024,38.505562,0.333748,4.106049,0.511271,0.285375,0.847376,0.551517,1.357987,0.05216,0.922529,3.600155,82.389999,89.010002,22.1,81.910004,84.75,82.620003,87.589996,88.769997,84.519997,82.389999,85.93,"""1""","""12""","""121""","""1212"""


# 7. Build SCB SSYK4 Employment Counts

Aggregate the SCB counts by year and the 4-digit SSYK Level.

In [9]:
print(scb_lazy_lf.head(5).collect())

shape: (5, 8)
┌───────┬───────────┬────────────────────┬─────────────┬────────────────────┬───────┬──────┬───────┐
│ level ┆ ssyk_code ┆ occupation         ┆ county_code ┆ county             ┆ sex   ┆ year ┆ value │
│ ---   ┆ ---       ┆ ---                ┆ ---         ┆ ---                ┆ ---   ┆ ---  ┆ ---   │
│ str   ┆ str       ┆ str                ┆ str         ┆ str                ┆ str   ┆ i64  ┆ i64   │
╞═══════╪═══════════╪════════════════════╪═════════════╪════════════════════╪═══════╪══════╪═══════╡
│ SSYK4 ┆ 2343      ┆ Preschool teachers ┆ 24          ┆ Västerbotten       ┆ men   ┆ 2024 ┆ 114   │
│       ┆           ┆                    ┆             ┆ county             ┆       ┆      ┆       │
│ SSYK4 ┆ 4113      ┆ Back office-staff  ┆ 17          ┆ Värmland county    ┆ women ┆ 2017 ┆ 6     │
│ SSYK4 ┆ 5311      ┆ Child care workers ┆ 18          ┆ Örebro county      ┆ women ┆ 2021 ┆ 2377  │
│ SSYK4 ┆ 4212      ┆ Debt-collectors,   ┆ 05          ┆ Östergötland       ┆

In [10]:
scb_lazy_lf_level4 = (
    scb_lazy_lf
        .filter(pl.col("level") == "SSYK4")
        .group_by(["ssyk_code", "year"])
        .agg(pl.col("value").sum().alias("total_count"))
)

scb_lazy_lf_level4.limit(5).collect()

ssyk_code,year,total_count
str,i64,i64
"""5151""",2021,3015
"""2672""",2016,1186
"""7523""",2014,6579
"""3515""",2021,2225
"""3360""",2023,18881


# 8. Merge DAIOE with SCB SSYK2012 at Level 4 Counts

Join by `year` and the `ssyk_code` the two tables.

This table is used in the aggregation of the daioe metrics.

In [11]:
daioe_scb_years = daioe_lazy_lf_extended\
    .join(
        scb_lazy_lf_level4,
        left_on=["year", "code_4"],
        right_on=["year", "ssyk_code"],
        how="left",
    )

daioe_scb_years.limit(10).collect()


year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4,total_count
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,i64
2014,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",861
2015,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",1536
2016,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",1359
2017,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",1183
2018,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",1108
2019,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",1871
2020,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",2286
2021,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",2331
2022,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110""",2572


In [12]:
# DAIOE codes with no SCB match
daioe_scb_years_unmatched = daioe_lazy_lf_extended\
    .join(
        scb_lazy_lf_level4,
        left_on=["year", "code_4"],
        right_on=["year", "ssyk_code"],
        how="anti"
    )

daioe_scb_years_unmatched.limit(10).collect()

year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str


# 9. Validate the SCB-DAIOE Level 4 Merge

Run quick schema and shape checks on the filtered joined frame before aggregation.

In [13]:
daioe_scb_years.collect_schema()

Schema([('year', Int64),
        ('daioe_allapps', Float64),
        ('daioe_stratgames', Float64),
        ('daioe_videogames', Float64),
        ('daioe_imgrec', Float64),
        ('daioe_imgcompr', Float64),
        ('daioe_imggen', Float64),
        ('daioe_readcompr', Float64),
        ('daioe_lngmod', Float64),
        ('daioe_translat', Float64),
        ('daioe_speechrec', Float64),
        ('daioe_genai', Float64),
        ('pctl_rank_allapps', Float64),
        ('pctl_rank_stratgames', Float64),
        ('pctl_rank_videogames', Float64),
        ('pctl_rank_imgrec', Float64),
        ('pctl_rank_imgcompr', Float64),
        ('pctl_rank_imggen', Float64),
        ('pctl_rank_readcompr', Float64),
        ('pctl_rank_lngmod', Float64),
        ('pctl_rank_translat', Float64),
        ('pctl_rank_speechrec', Float64),
        ('pctl_rank_genai', Float64),
        ('code_1', String),
        ('code_2', String),
        ('code_3', String),
        ('code_4', String),
        ('tot

# 10. Aggregation of the DAIOEs with the employment data 

Here, I begin the manual steps of aggregating the original daioe's metrics from the level 4 to level 1.



## 10(a). Utility Helper Function

Aggregate SSYK4-1 with a reusable function and add within-year percentiles.

In [14]:
from dataclasses import dataclass


@dataclass
class AggConfig:
    weight_col: str = "total_count"
    prefix: str = "daioe_"
    add_percentiles: bool = True
    pct_scale: int = 100
    descending: bool = False
    # Exposure level boundaries
    pct_l1: int = 20
    pct_l2: int = 40
    pct_l3: int = 60
    pct_l4: int = 80




In [15]:
def aggregate_daioe_level(
    lf: pl.LazyFrame,
    code_col: str,
    level_label: str,
    cfg: AggConfig | None = None,
) -> pl.LazyFrame:
    """
    Aggregate DAIOE metrics by SSYK level with optional weighted averages and percentile ranks.

    Args:
        lf:           Input LazyFrame.
        code_col:     Column name for SSYK code (e.g. "code_1", "code_2").
        level_label:  Label for the aggregation level (e.g. "ssyk1", "ssyk2").
        cfg:          Aggregation configuration (see AggConfig). Defaults to AggConfig().

    """
    if cfg is None:
        cfg = AggConfig()

    daioe_cols = [c for c in lf.collect_schema().names() if c.startswith(cfg.prefix)]
    if not daioe_cols:
        msg = f"No DAIOE columns found with prefix {cfg.prefix!r}"
        raise ValueError(msg)

    w = pl.col(cfg.weight_col)
    mean_exprs = [pl.col(c).mean().alias(f"{c}_avg") for c in daioe_cols]
    weighted_avg_exprs = [
        pl.when(
            (denom := pl.when(pl.col(c).is_not_null()).then(w).otherwise(None).sum()) > 0
        )
        .then((pl.col(c) * w).sum() / denom)
        .otherwise(None)
        .alias(f"{c}_wavg")
        for c in daioe_cols
    ]

    out = (
        lf
        .group_by(["year", code_col])
        .agg(w.sum().alias("weight_sum"), *mean_exprs, *weighted_avg_exprs)
        .with_columns(pl.lit(level_label).alias("level"))
        .rename({code_col: "ssyk_code"})
    )

    if not cfg.add_percentiles:
        return out

    group_keys = ["year", "level"]
    n_expr = pl.len().over(group_keys)
    rank_expr = (
        pl.col(f"^{cfg.prefix}.*_(avg|wavg)$")
        .rank(method="average", descending=cfg.descending)
        .over(group_keys)
    )

    return out.with_columns(
        (
            pl.when(n_expr > 1)
            .then((rank_expr - 1) / (n_expr - 1))
            .otherwise(0.0)
            * cfg.pct_scale
        ).name.prefix("pctl_"),
    )

## 10(b). Aggregating the DAIOEs 

Aggregate SSYK4-1 with a reusable function and add within-year percentiles.

In [16]:
levels_daioe = {
    "code_4": "SSYK4",
    "code_3": "SSYK3",
    "code_2": "SSYK2",
    "code_1": "SSYK1",
}

aggregated = [
    aggregate_daioe_level(daioe_scb_years, col, label)
    for col, label in levels_daioe.items()
]

daioe_all_levels = (
    pl.concat(aggregated)
    .sort(["level", "year", "ssyk_code"])
)

daioe_all_levels.limit(10).collect()

year,ssyk_code,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,level,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2014,"""0""",10382,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""SSYK1""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2014,"""1""",270311,4.831857,0.202699,1.804139,0.145464,null,null,0.003346,0.013267,null,0.044049,0.013267,4.930529,0.208521,1.829927,0.150019,null,null,0.003454,0.013649,null,0.04487,0.013649,"""SSYK1""",11.111111,55.555556,0.0,55.555556,null,null,66.666667,66.666667,null,66.666667,66.666667,11.111111,55.555556,0.0,55.555556,null,null,66.666667,66.666667,null,66.666667,66.666667
2014,"""2""",958595,5.350139,0.22427,2.02902,0.158891,null,null,0.00367,0.014341,null,0.046376,0.014341,5.342221,0.226313,2.002547,0.158653,null,null,0.003723,0.014687,null,0.047908,0.014687,"""SSYK1""",44.444444,88.888889,11.111111,77.777778,null,null,88.888889,77.777778,null,77.777778,77.777778,44.444444,77.777778,11.111111,77.777778,null,null,88.888889,77.777778,null,77.777778,77.777778
2014,"""3""",547158,5.456573,0.212926,2.212758,0.153986,null,null,0.003013,0.011972,null,0.042044,0.011972,5.439659,0.216966,2.148252,0.157487,null,null,0.003217,0.012836,null,0.044524,0.012836,"""SSYK1""",55.555556,66.666667,44.444444,66.666667,null,null,55.555556,55.555556,null,55.555556,55.555556,66.666667,66.666667,33.333333,66.666667,null,null,55.555556,55.555556,null,55.555556,55.555556
2014,"""4""",336357,5.672183,0.220959,2.190429,0.160599,null,null,0.003519,0.014595,null,0.055095,0.014595,5.907369,0.232485,2.302672,0.167467,null,null,0.003611,0.014823,null,0.053938,0.014823,"""SSYK1""",88.888889,77.777778,33.333333,88.888889,null,null,77.777778,88.888889,null,88.888889,88.888889,88.888889,88.888889,44.444444,88.888889,null,null,77.777778,88.888889,null,88.888889,88.888889
2014,"""5""",889837,4.7938,0.169823,2.070333,0.119541,null,null,0.002216,0.009206,null,0.036409,0.009206,4.676852,0.167826,2.016777,0.116503,null,null,0.002179,0.009016,null,0.034695,0.009016,"""SSYK1""",0.0,11.111111,22.222222,11.111111,null,null,44.444444,44.444444,null,44.444444,44.444444,0.0,11.111111,22.222222,0.0,null,null,44.444444,44.444444,null,44.444444,44.444444
2014,"""6""",31111,5.12747,0.173135,2.40164,0.125359,null,null,0.00167,0.006697,null,0.026764,0.006697,5.204076,0.17395,2.448156,0.126372,null,null,0.001697,0.006771,null,0.027124,0.006771,"""SSYK1""",33.333333,22.222222,55.555556,22.222222,null,null,33.333333,22.222222,null,22.222222,22.222222,33.333333,22.222222,55.555556,22.222222,null,null,33.333333,33.333333,null,33.333333,33.333333
2014,"""7""",365544,5.497783,0.18516,2.645227,0.129784,null,null,0.001618,0.006395,null,0.024655,0.006395,5.382421,0.183274,2.576241,0.128829,null,null,0.001604,0.006296,null,0.024398,0.006296,"""SSYK1""",66.666667,44.444444,77.777778,33.

In [17]:
print(
    daioe_all_levels
    .group_by("level")
    .len()
    .collect(),
)

shape: (4, 2)
┌───────┬──────┐
│ level ┆ len  │
│ ---   ┆ ---  │
│ str   ┆ u32  │
╞═══════╪══════╡
│ SSYK2 ┆ 506  │
│ SSYK3 ┆ 1628 │
│ SSYK1 ┆ 110  │
│ SSYK4 ┆ 4719 │
└───────┴──────┘


# 11. Build 1-5 Level of EXposure Columns

Create `daioe_<index>_Level_Exposure` columns from weighted percentile ranks (`pctl_daioe_*_wavg`) using quintile-style bins.

In [18]:
# Convert weighted percentile ranks (0..100) into 1-5 exposure levels
pct_cols = [
    c
    for c in daioe_all_levels.collect_schema().names()
    if c.startswith("pctl_daioe_") and c.endswith("_wavg")
]

exposure_exprs = [
    pl.when(pl.col(col).is_null())
    .then(None)
    .when(pl.col(col) <= AggConfig.pct_l1).then(1)
    .when(pl.col(col) <= AggConfig.pct_l2).then(2)
    .when(pl.col(col) <= AggConfig.pct_l3).then(3)
    .when(pl.col(col) <= AggConfig.pct_l4).then(4)
    .otherwise(5)
    .cast(pl.Int8)
    .alias(f"daioe_{col[len('pctl_daioe_'):-len('_wavg')]}_Level_Exposure")
    for col in pct_cols
]

daioe_all_levels = daioe_all_levels.with_columns(exposure_exprs)
print([c for c in daioe_all_levels.collect_schema().names() if c.endswith("_Level_Exposure")])

['daioe_allapps_Level_Exposure', 'daioe_stratgames_Level_Exposure', 'daioe_videogames_Level_Exposure', 'daioe_imgrec_Level_Exposure', 'daioe_imgcompr_Level_Exposure', 'daioe_imggen_Level_Exposure', 'daioe_readcompr_Level_Exposure', 'daioe_lngmod_Level_Exposure', 'daioe_translat_Level_Exposure', 'daioe_speechrec_Level_Exposure', 'daioe_genai_Level_Exposure']


# 12. Merging the data 

After all the data processes are done we now merge `scb_lazy_lf_changes` with the `daioe_all_levels`.

In [19]:
daioe_all_levels.collect_schema()

Schema([('year', Int64),
        ('ssyk_code', String),
        ('weight_sum', Int64),
        ('daioe_allapps_avg', Float64),
        ('daioe_stratgames_avg', Float64),
        ('daioe_videogames_avg', Float64),
        ('daioe_imgrec_avg', Float64),
        ('daioe_imgcompr_avg', Float64),
        ('daioe_imggen_avg', Float64),
        ('daioe_readcompr_avg', Float64),
        ('daioe_lngmod_avg', Float64),
        ('daioe_translat_avg', Float64),
        ('daioe_speechrec_avg', Float64),
        ('daioe_genai_avg', Float64),
        ('daioe_allapps_wavg', Float64),
        ('daioe_stratgames_wavg', Float64),
        ('daioe_videogames_wavg', Float64),
        ('daioe_imgrec_wavg', Float64),
        ('daioe_imgcompr_wavg', Float64),
        ('daioe_imggen_wavg', Float64),
        ('daioe_readcompr_wavg', Float64),
        ('daioe_lngmod_wavg', Float64),
        ('daioe_translat_wavg', Float64),
        ('daioe_speechrec_wavg', Float64),
        ('daioe_genai_wavg', Float64),
        

In [20]:
scb_lazy_lf_changes.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('occupation', String),
        ('county_code', String),
        ('county', String),
        ('sex', String),
        ('year', Int64),
        ('emp_count', Int64),
        ('chg_1y', Int64),
        ('chg_3y', Int64),
        ('chg_5y', Int64),
        ('pct_chg_1y', Float64),
        ('pct_chg_3y', Float64),
        ('pct_chg_5y', Float64)])

In [21]:
final_merge = (
    scb_lazy_lf_changes
    .join(
        daioe_all_levels,
        left_on = ["year", "ssyk_code"],
        right_on = ["year", "ssyk_code"],
        how = "left",
    ).drop(pl.col("level_right"))
)

In [22]:
final_merge.limit(10).collect()

level,ssyk_code,occupation,county_code,county,sex,year,emp_count,chg_1y,chg_3y,chg_5y,pct_chg_1y,pct_chg_3y,pct_chg_5y,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure
str,str,str,str,str,str,i64,i64,i64,i64,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""SSYK3""","""123""","""Administration and planning ma…","""01""","""Stockholm county""","""women""",2016,1478,117,null,null,8.59662,null,null,10666,14.388259,0.26857,2.810883,0.194944,0.051114,0.350965,0.147457,0.040884,0.010469,0.299156,0.571254,14.388259,0.26857,2.810883,0.194944,0.051114,0.350965,0.147457,0.040884,0.010469,0.299156,0.571254,68.027211,61.22449,27.210884,68.027211,72.108844,69.387755,71.428571,72.108844,78.911565,80.272109,71.428571,69.387755,62.585034,26.530612,69.387755,73.469388,70.068027,68.707483,71.428571,78.231293,82.312925,71.428571,4,4,2,4,4,4,4,4,4,5,4
"""SSYK2""","""51""","""Service workers""","""04""","""Södermanland county""","""women""",2024,1637,-94,307,301,-5.430387,23.082707,22.52994,160088,29.323046,0.234816,4.758632,0.372868,0.194541,0.572174,0.294871,0.757082,0.031658,0.650151,2.181008,28.168158,0.233271,5.040811,0.360284,0.182443,0.523468,0.260599,0.667578,0.027348,0.574705,1.9577,33.333333,28.888889,40.0,20.0,35.555556,35.555556,40.0,40.0,40.0,44.444444,37.777778,24.444444,28.888889,44.444444,11.111111,22.222222,17.777778,33.333333,37.777778,37.777778,37.777778,33.333333,2,2,3,1,2,1,2,2,2,2,2
"""SSYK4""","""2143""","""Engineering professionals in e…","""09""","""Gotland county""","""women""",2016,4,1,null,null,33.333333,null,null,26633,18.06826,0.365276,3.619483,0.245165,0.0642,0.471704,0.190191,0.049799,0.011428,0.304197,0.751426,18.06826,0.365276,3.619483,0.245165,0.0642,0.471704,0.190191,0.049799,0.011428,0.304197,0.751426,94.392523,93.691589,69.859813,90.420561,92.28972,92.056075,90.654206,87.850467,84.813084,78.738318,92.056075,94.392523,93.691589,69.859813,90.420561,92.28972,92.056075,90.654206,87.850467,84.813084,78.738318,92.056075,5,5,4,5,5,5,5,5,5,4,5
"""SSYK3""","""154""","""Managers and leaders within re…","""05""","""Östergötland county""","""men""",2020,19,-1,6,2,-5.0,46.153846,11.764706,795,22.157991,0.233331,3.107001,0.312597,0.081067,0.504441,0.207073,0.357876,0.042944,0.433057,1.504555,22.157991,0.233331,3.107001,0.312597,0.081067,0.504441,0.207073,0.357876,0.042944,0.433057,1.504555,32.653061,25.85034,0.0,36.734694,53.061224,56.462585,61.904762,68.707483,73.469388,73.469388,63.265306,31.972789,26.530612,0.0,36.734694,53.741497,57.142857,61.22449,68.027211,73.469388,72.108844,62.585034,2,2,1,2,3,3,4,4,4,4,4
"""SSYK2""","""2

# 13. Export the Final Dataset

rite the merged output to parquet for upstream use.

In [45]:
output_path = DATA_DIR / "daioe_scb_years_all_levels.parquet"

final_merge.sink_parquet(output_path)